# Metal Foam Porosity · Image Segmentation

**From circle detection to pore-region reconstruction**  
Python · Computer Vision · Scientific Imaging · Image Analysis

This reproducible notebook detects pores in 2D cross-sections of a metallic foam using a **multi-scale circular Hough transform**, suppresses duplicate detections, reconstructs irregular pore boundaries and estimates **2D area porosity**.

**Key contributions:** small-pore detection, IoU-based non-maximum suppression, convex specimen masking, seed-based region growing, connected-component grouping and quantitative analysis.

> **Scope:** exploratory research pipeline, not a validated metrology instrument. Images are median slices extracted from TIFF stacks, not a complete 3D segmentation. A separate [experimental log](02_experimental_journey.ipynb) documents the iterations and failures.

## 0. Setup and reproducibility

The installation line below is deliberately kept in the notebook so that a new user can start with a fresh Jupyter environment. Execute it **once**, then restart the kernel if packages were newly installed.

Place your `.tif` files in `input/` (see README). The original scan data are not distributed with this repository.

In [ ]:
%pip install numpy pandas matplotlib scipy scikit-image tifffile

## 1. Data loading

Each input file is a TIFF stack. We extract its middle axial slice and normalize it to 8-bit. This reproduces the processing used during development; note that independent normalization may affect between-scan intensity comparisons.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile

from skimage.feature import canny
from skimage.transform import hough_circle, hough_circle_peaks
from skimage.filters import gaussian, threshold_otsu
from skimage.measure import label, regionprops
from skimage.morphology import (
    binary_closing, binary_erosion, disk, convex_hull_image,
    remove_small_objects
)
from skimage.segmentation import flood
from scipy.ndimage import binary_fill_holes

DATA_DIR = Path("input")
scan_paths = sorted(DATA_DIR.glob("*.tif"))
assert scan_paths, f"Aucun TIFF trouvé dans {DATA_DIR.resolve()}"
print(f"{len(scan_paths)} coupes trouvées")
for p in scan_paths:
    print(" -", p.name)


In [ ]:
def load_middle_image(path):
    stack = tifffile.imread(path)
    middle = stack[stack.shape[0] // 2] if stack.ndim == 3 else stack
    max_value = float(np.max(middle))
    if max_value <= 0:
        raise ValueError(f"Image vide : {path}")
    return np.clip(255 * middle.astype(float) / max_value, 0, 255).astype(np.uint8)

example = load_middle_image(scan_paths[-1])
plt.figure(figsize=(5, 5))
plt.imshow(example, cmap="gray")
plt.title("Exemple : coupe médiane")
plt.axis("off")
plt.show()


## 2. Multi-scale pore candidate detection

Canny edges feed two Hough searches: **radii 4–18 px** (small pores) and **20–88 px** (large pores). Candidates below the Hough score threshold **0.20** or local contrast **20** are rejected. Greedy **IoU-NMS (0.30)** suppresses overlapping circle hypotheses.

The Hough circles are **seeds**, not the final pore geometry.

In [ ]:
def filter_by_contrast(image, xs, ys, radii, scores, min_contrast=20):
    h, w = image.shape
    yy, xx = np.ogrid[:h, :w]
    kept = []
    for i, (x, y, r) in enumerate(zip(xs, ys, radii)):
        d2 = (xx - x)**2 + (yy - y)**2
        inside = d2 <= (0.8*r)**2
        outside = ((d2 >= (1.1*r)**2) & (d2 <= (1.4*r)**2))
        if inside.any() and outside.any():
            if image[outside].mean() - image[inside].mean() >= min_contrast:
                kept.append(i)
    return np.asarray(kept, dtype=int)

def circle_iou(x1, y1, r1, x2, y2, r2):
    d = np.hypot(x2-x1, y2-y1)
    if d >= r1+r2:
        return 0.0
    a1, a2 = np.pi*r1**2, np.pi*r2**2
    if d <= abs(r1-r2):
        intersection = np.pi * min(r1, r2)**2
    else:
        alpha = np.arccos(np.clip((d*d+r1*r1-r2*r2)/(2*d*r1), -1, 1))
        beta = np.arccos(np.clip((d*d+r2*r2-r1*r1)/(2*d*r2), -1, 1))
        term = max(0, (-d+r1+r2)*(d+r1-r2)*(d-r1+r2)*(d+r1+r2))
        intersection = r1*r1*alpha + r2*r2*beta - 0.5*np.sqrt(term)
    return intersection/(a1+a2-intersection)

def nms_iou(scores, xs, ys, rs, threshold=0.3):
    kept = []
    for i in np.argsort(scores)[::-1]:
        if all(circle_iou(xs[i],ys[i],rs[i],xs[j],ys[j],rs[j]) <= threshold
               for j in kept):
            kept.append(i)
    return np.asarray(kept, dtype=int)


In [ ]:
def detect_multiscale_candidates(image):
    edges = canny(image, sigma=2)
    all_scores, all_xs, all_ys, all_rs = [], [], [], []
    for radii_range in (np.arange(4, 20, 2), np.arange(20, 90, 4)):
        hough_result = hough_circle(edges, radii_range)
        scores, xs, ys, rs = hough_circle_peaks(
            hough_result, radii_range, total_num_peaks=300,
            min_xdistance=1, min_ydistance=1
        )
        valid = scores >= 0.20
        scores, xs, ys, rs = scores[valid], xs[valid], ys[valid], rs[valid]
        kept = filter_by_contrast(image, xs, ys, rs, scores, min_contrast=20)
        all_scores.extend(scores[kept])
        all_xs.extend(xs[kept])
        all_ys.extend(ys[kept])
        all_rs.extend(rs[kept])
    scores = np.asarray(all_scores)
    xs = np.asarray(all_xs, dtype=int)
    ys = np.asarray(all_ys, dtype=int)
    rs = np.asarray(all_rs, dtype=int)
    if len(scores):
        kept = nms_iou(scores, xs, ys, rs, threshold=0.3)
    else:
        kept = np.asarray([], dtype=int)
    return {"scores": scores[kept], "xs": xs[kept],
            "ys": ys[kept], "radii": rs[kept]}


## 3. Specimen geometry and pore reconstruction

We identify the bright specimen using smoothing, Otsu thresholding and the largest connected component. Its **convex hull** recovers peripheral pores that would otherwise be excluded by indentations in the specimen mask.

Each retained circle seeds a **local intensity flood** (`tolerance=18`) constrained to a radius of `1.6 × r`. We union the resulting masks, remove very small components, fill internal holes, and label the connected regions.

**Caution:** connected-component grouping can also merge physically distinct pores when an erroneous thin bridge is formed.

In [ ]:
def create_sample_mask(image, margin=0):
    smooth = gaussian(image.astype(float), sigma=8)
    sample = smooth > threshold_otsu(smooth)
    labels = label(sample)
    regions = regionprops(labels)
    if not regions:
        raise ValueError("Échantillon non détecté")
    sample = labels == max(regions, key=lambda r: r.area).label
    sample = binary_closing(sample, footprint=disk(8))
    sample = binary_fill_holes(sample)
    if margin > 0:
        sample = binary_erosion(sample, footprint=disk(margin))
    return sample

def convex_sample_mask(image):
    return convex_hull_image(create_sample_mask(image, margin=0))


In [ ]:
def reconstruct_all_pores(image, detection, tolerance=18):
    smooth = gaussian(image, sigma=1.5, preserve_range=True)
    h, w = image.shape
    yy, xx = np.ogrid[:h, :w]
    combined = np.zeros((h, w), dtype=bool)
    for x, y, r in zip(detection["xs"], detection["ys"], detection["radii"]):
        x, y, r = int(x), int(y), float(r)
        if not (0 <= x < w and 0 <= y < h):
            continue
        region = flood(smooth, (y, x), tolerance=tolerance)
        local_area = (xx-x)**2 + (yy-y)**2 <= (1.6*r)**2
        combined |= (region & local_area)
    combined = remove_small_objects(combined, min_size=15)
    return combined

def segment_pores(image, tolerance=18):
    candidates = detect_multiscale_candidates(image)
    sample = convex_sample_mask(image)
    valid = [j for j, (x,y) in enumerate(zip(candidates["xs"], candidates["ys"]))
             if 0 <= int(x) < image.shape[1] and 0 <= int(y) < image.shape[0]
             and sample[int(y), int(x)]]
    detection = {key: values[valid] for key, values in candidates.items()}
    mask = reconstruct_all_pores(image, detection, tolerance=tolerance)
    mask = binary_fill_holes(mask)
    # Même logique que l'expérience 25 : ne pas couper le masque reconstruit
    # par le masque d'échantillon ; l'intersection sera faite pour les statistiques.
    labeled = label(mask)
    return {"mask": mask, "labeled": labeled, "detection": detection,
            "sample_mask": sample, "n_circles": len(detection["radii"]),
            "n_regions": int(labeled.max())}


## 4. Batch processing

The same parameters are applied to all available TIFF stacks. The program retains each image and its segmentation in memory for plots and statistics. Processing may take several minutes.

In [ ]:
all_images = []
all_results = []
for i, path in enumerate(scan_paths):
    image = load_middle_image(path)
    result = segment_pores(image)
    all_images.append(image)
    all_results.append(result)
    print(f"Image {i:02d} : {result['n_circles']} cercles → {result['n_regions']} régions")


## 5. Visual quality review

Inspect the green outlines against the raw images. Typical failure modes: missed micro-pores, artificially merged pores and region growing into brighter material. A visually good contour is **not** equivalent to quantitative validation.

In [ ]:
fig, axes = plt.subplots(len(all_images), 2, figsize=(11, 4.4*len(all_images)),
                         squeeze=False)
for i, (image, result) in enumerate(zip(all_images, all_results)):
    axes[i, 0].imshow(image, cmap="gray")
    axes[i, 0].set_title(f"Image {i:02d} — originale")
    axes[i, 1].imshow(image, cmap="gray")
    axes[i, 1].contour(result["mask"].astype(float), levels=[0.5],
                       colors="lime", linewidths=0.8)
    axes[i, 1].set_title(f"Image {i:02d} — {result['n_regions']} régions")
    for ax in axes[i]:
        ax.axis("off")
plt.tight_layout()
plt.show()


## 6. Circle morphology and spatial distribution

The following metrics refer to **Hough circles retained after filtering**, not to final connected pore regions. Diameter is `2 × radius` in pixels. Origin refers to the `(x, y)` center of each circle.

In [ ]:
circle_stats = []
for i, result in enumerate(all_results):
    d = result["detection"]
    radii = np.asarray(d["radii"], dtype=float)
    circle_stats.append({
        "image": i, "circles": len(radii),
        "radius_min_px": radii.min() if len(radii) else np.nan,
        "radius_max_px": radii.max() if len(radii) else np.nan,
        "radius_mean_px": radii.mean() if len(radii) else np.nan,
        "radius_std_px": radii.std() if len(radii) else np.nan
    })
df_circles = pd.DataFrame(circle_stats)
display(df_circles.round(2))

In [ ]:
from matplotlib.patches import Circle

fig, axes = plt.subplots(2, 5, figsize=(17, 7))
for i, result in enumerate(all_results[:10]):
    radii = np.asarray(result["detection"]["radii"])
    ax = axes.flat[i]
    ax.hist(2*radii, bins=np.arange(0, 190, 10), edgecolor="white")
    ax.set(title=f"Slice {i:02d}", xlabel="Diameter (px)", ylabel="Count")
for ax in axes.flat[len(all_results[:10]):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 5, figsize=(17, 8))
for i, (image, result) in enumerate(zip(all_images[:10], all_results[:10])):
    ax = axes.flat[i]
    ax.imshow(image, cmap="gray")
    d = result["detection"]
    ax.scatter(d["xs"], d["ys"], s=8, c="lime", alpha=.75)
    ax.set(title=f"Circle centers — slice {i:02d}")
    ax.axis("off")
for ax in axes.flat[len(all_images[:10]):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Porosity measurement

For each 2D slice, we calculate:

\[
\text{Area porosity (\%)} = 100 \frac{|\text{pore mask} \cap \text{specimen mask}|}{|\text{specimen mask}|}
\]

The unit is **pixel area**; no physical calibration is assumed. The output is a segmentation-derived **2D area fraction**, not a validated 3D volume fraction.

In [ ]:
records = []
for i, result in enumerate(all_results):
    sample = result["sample_mask"]
    pore_inside = result["mask"] & sample
    sample_area = int(sample.sum())
    pore_area = int(pore_inside.sum())
    areas = [r.area for r in regionprops(result["labeled"])]
    records.append({
        "Image": i,
        "Cercles Hough": result["n_circles"],
        "Régions détectées": result["n_regions"],
        "Surface pores (px²)": pore_area,
        "Surface échantillon (px²)": sample_area,
        "Porosité 2D (%)": 100*pore_area/sample_area,
        "Surface moyenne région (px²)": float(np.mean(areas)) if areas else 0,
        "Surface max région (px²)": float(max(areas)) if areas else 0
    })
df_porosity = pd.DataFrame(records)
display(df_porosity.round(2))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df_porosity["Image"], df_porosity["Porosité 2D (%)"], marker="o")
axes[0].set(title="Fraction surfacique détectée", xlabel="Index de coupe",
            ylabel="Porosité 2D (%)")
axes[1].plot(df_porosity["Image"], df_porosity["Régions détectées"], marker="o")
axes[1].set(title="Régions détectées", xlabel="Index de coupe",
            ylabel="Nombre de régions")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 8. Interpretation and limitations

**Development observations (10 sample slices):** the measured area fraction increased from **4.70%** (slice 00) to approximately **29.59%** (slice 09); the peak observed was **29.80%** (slice 08). The number of connected regions generally decreased in the later slices as detected region sizes increased. These are **observations of different slices**, not evidence of temporal growth or physical pore coalescence.

**Known limitations:** missed small pores in early slices; a false region bridge/overgrowth in slice 02; occasional pore merges in slice 04 and eroded boundaries in slice 05; sensitivity to threshold parameters; no ground-truth labels. Some 2D area fractions may be biased in either direction.

**Next steps:** annotated validation masks; precision/recall and intersection-over-union metrics; parameter sensitivity; independent specimen-boundary assessment; calibrated pixel scale; full 3D processing.

For the full trial-and-error history, open **`02_experimental_journey.ipynb`**.